In [ ]:
import csv

# Convert raw tab-separated file to CSV
with open("data/raw_data/data.txt") as fin, open("data/raw_data/data.csv", "w", newline="") as fout:
    reader = csv.reader(fin, delimiter="\t")
    writer = csv.writer(fout)
    writer.writerow(["collected_at", "line", "transport_mode", "planned_time", "estimated_time"])
    writer.writerows(reader)

print("Converted data.txt → data/raw_data/data.csv")


,collected_at,service_journey_gid,line,line_short_name,line_designation,line_name,transport_mode,direction,stop_point_gid,stop_point_name,platform,realtime_platform,planned_time,estimated_time,best_time,is_cancelled,is_part_cancelled,departure_key
0,2026-03-08T21:12:32+01:00,9015014500605551,6,6,6,Spårvagn 6,tram,Kortedala via Centralstationen,9022014001960001,"Chalmers, Göteborg",A,NaN,2026-03-08T21:12:00.0000000+01:00,2026-03-08T21:14:00.0000000+01:00,2026-03-08T21:14:00.0000000+01:00,False,False,9015014500605551|9022014001960001|2026-03-08T2...
1,2026-03-08T21:12:32+01:00,9015014506400709,64,64,64,Buss 64,bus,"Heden, Påstigning fram",9022014001960003,"Chalmers, Göteborg",C,NaN,2026-03-08T21:15:00.0000000+01:00,2026-03-08T21:15:00.0000000+01:00,2026-03-08T21:15:00.0000000+01:00,False,False,9015014506400709|9022014001960003|2026-03-08T2...
2,2026-03-08T21:12:32+01:00,9015014506400710,64,64,64,Buss 64,bus,"Högsbohöjd, Påstigning fram",9022014001960004,"Chalmers, Göteborg",D,NaN,2026-03-08T21:18:00.0000000+01:00,2026-03-08T21:20:00.0000000+01:00,2026-03-08T21:20:00.0000000+01:00,False,False,9015014506400710|9022014001960004|2026-03-08T2...
3,2026-03-08T21:12:32+01:00,9015014500605602,6,6,6,Spårvagn 6,tram,Länsmansgården via Sahlgrenska,9022014001960002,"Chalmers, Göteborg",B,NaN,2026-03-08T21:23:00.0000000+01:00,2026-03-08T21:29:00.0000000+01:00,2026-03-08T21:29:00.0000000+01:00,False,False,9015014500605602|9022014001960002|2026-03-08T2...
4,2026-03-08T21:12:32+01:00,9015014500605575,6,6,6,Spårvagn 6,tram,Kortedala via Centralstationen,9022014001960001,"Chalmers, Göteborg",A,NaN,2026-03-08T21:27:00.0000000+01:00,2026-03-08T21:30:00.0000000+01:00,2026-03-08T21:30:00.0000000+01:00,False,False,9015014500605575|9022014001960001|2026-03-08T2...


In [62]:
import pandas as pd

# ── Direction lookup: (line, HH:MM) → direction label ────────────────────────
DIRECTION_FILES = {
    ("data/cleaned_data/line_6__kortedala_scheduled.csv",  6):  "Kortedala via Centralstationen",
    ("data/cleaned_data/line_6_varmfront_scheduled.csv",   6):  "Länsmansgården via Sahlgrenska",
    ("data/cleaned_data/line_64_heden_scheduled.csv",      64): "Heden",
    ("data/cleaned_data/line_64_axel_scheduled.csv",       64): "Högsbohöjd",
}

lookup = {}
for (path, line_num), direction in DIRECTION_FILES.items():
    dir_df = pd.read_csv(path)
    for t in dir_df["departure_time"]:
        lookup[(line_num, str(t)[:5])] = direction

# ── Load raw data, filter lines 6 & 64 ───────────────────────────────────────
raw = pd.read_csv("data/raw_data/data.csv")
raw = raw[raw["line"].isin([6, 64])].copy()

# ── Parse times to HH:MM:SS ──────────────────────────────────────────────────
raw["planned_time"]   = pd.to_datetime(raw["planned_time"]).dt.strftime("%H:%M:%S")
raw["estimated_time"] = pd.to_datetime(raw["estimated_time"]).dt.strftime("%H:%M:%S")

# ── Assign direction via lookup on (line, HH:MM) ─────────────────────────────
raw["direction"] = raw.apply(
    lambda r: lookup.get((r["line"], r["planned_time"][:5]), "unknown"),
    axis=1
)

# ── Deduplicate: unique departure = (line, direction, planned_time), keep last ─
final = (
    raw
    .drop_duplicates(subset=["line", "direction", "planned_time"], keep="last")
    .reset_index(drop=True)
)

# ── Compute delay in minutes ──────────────────────────────────────────────────
def to_minutes(t: str) -> int:
    h, m, _ = t.split(":")
    return int(h) * 60 + int(m)

final["delay_minutes"] = (
    final["estimated_time"].apply(to_minutes) - final["planned_time"].apply(to_minutes)
)

# ── Keep only the columns we care about ──────────────────────────────────────
final = final[["line", "direction", "planned_time", "estimated_time", "delay_minutes"]]

# ── Save ──────────────────────────────────────────────────────────────────────
out = "data/cleaned_data/final_departures.csv"
final.to_csv(out, index=False)
print(f"Saved {len(final)} rows → {out}")
print(f"\nunknown direction count: {(final['direction'] == 'unknown').sum()}")
print(f"\nDirections:\n{final.groupby(['line','direction']).size().to_string()}")
final


Saved 68 rows → data/cleaned_data/final_departures.csv

unknown direction count: 1

Directions:
line  direction                     
6     Kortedala via Centralstationen    16
      Länsmansgården via Sahlgrenska    19
64    Heden                             13
      Högsbohöjd                        19
      unknown                            1


,line,direction,planned_time,estimated_time,delay_minutes
0,6,Länsmansgården via Sahlgrenska,06:31:00,06:33:00,2
1,64,Högsbohöjd,06:28:00,06:35:00,7
2,64,Heden,06:36:00,06:35:00,-1
3,6,Länsmansgården via Sahlgrenska,06:19:00,06:38:00,19
4,6,Kortedala via Centralstationen,06:35:00,06:39:00,4
...,...,...,...,...,...
63,64,Högsbohöjd,09:19:00,09:19:00,0
64,6,Kortedala via Centralstationen,09:23:00,09:24:00,1
65,64,Heden,09:25:00,09:25:00,0
66,6,Länsmansgården via Sahlgrenska,09:22:00,09:27:00,5
